#### Data Wrangling — Kualitas Udara & Dampak Kesehatan
Alur: **Gathering → Assessing → Cleaning → Integrasi → Simpan**

#### Setup

In [5]:
%pip install pandas numpy matplotlib seaborn

  Using cached pandas-3.1.0-cp312-cp312-win_amd64.whl.metadata (18 kB)
  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.68.0-cp312-cp312-win_amd64.whl.metadata (142 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
Using cached pandas-3.1.0-cp312-cp312-win_amd64.whl (10.7 MB)
Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl (12.6 MB)
Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl (234 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.68.0-cp312-cp312-win_amd64.whl (2.5 MB)
U

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\Mega Agustina\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\numpy\\_core\\tests\\test_records.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Mega Agustina\AppData\Local\Programs\Python\Python312\python.exe
3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]


In [3]:
import sys, subprocess
r = subprocess.run([sys.executable, "-m", "pip", "install", "pandas", "numpy", "matplotlib", "seaborn", "ipykernel"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
print(r.stderr[-1500:])

stina\appdata\roaming\python\python312\site-packages (from ipython>=7.23.1->ipykernel) (2.19.2)


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



In [4]:
import pandas as pd
print(pd.__version__)

3.1.0


In [5]:
import pandas as pd, numpy as np, glob, os
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA = "../data"
POLLUTANTS = ["pm25_ug_m3", "pm10_ug_m3", "so2_ug_m3", "co_ug_m3", "o3_ug_m3", "no2_ug_m3"]
METEO = ["temperature_c", "humidity_pct", "wind_speed_ms", "wind_direction_deg"]
HEALTH_COUNTS = ["resp_admissions_under5", "resp_admissions_over60", "asthma_emergency_cases",
                 "copd_admissions", "resp", "cvd", "death"]
HEALTH_WEATHER = ["temp_c", "dptp_c", "rhum_pct"]

cleaning_log = []
def log(step, n, note=""):
    cleaning_log.append({"step": step, "rows_affected": int(n), "note": note})
    print(f"[LOG] {step}: {int(n):,} {note}")

def audit(df, name):
    print(f"===== {name} | {df.shape[0]:,} baris x {df.shape[1]} kolom =====")
    print("Duplikat baris penuh:", df.duplicated().sum())
    na = df.isna().sum()
    print("Kolom dengan nilai kosong:")
    display(na[na > 0].to_frame("jumlah_kosong") if (na > 0).any() else "tidak ada")

In [6]:
# Gabung semua file AQMS, sambil mengecek apakah struktur kolomnya seragam
files = sorted(glob.glob(f"{DATA}/aqms/*_AQMS.csv"))
print(len(files), "file AQMS ditemukan")

frames, col_sets = [], {}
for f in files:
    df = pd.read_csv(f)
    name = os.path.basename(f)
    col_sets[name] = tuple(df.columns)
    df["source_file"] = name
    frames.append(df)

print("Jumlah versi struktur kolom:", len(set(col_sets.values())), "(idealnya 1)")
aqms_raw = pd.concat(frames, ignore_index=True)
del frames
print("Total baris AQMS:", f"{len(aqms_raw):,}")
aqms_raw.head()

27 file AQMS ditemukan
Jumlah versi struktur kolom: 2 (idealnya 1)
Total baris AQMS: 667,981


,measurement_id,sensor_id,timestamp_start,timestamp_end,pm25_ug_m3,pm10_ug_m3,so2_ug_m3,co_ug_m3,o3_ug_m3,no2_ug_m3,temperature_c,humidity_pct,wind_speed_ms,wind_direction_deg,qc_flag,source_file,pm25_ppm,pm10_ppm,so2_ppm,co_ppm,o3_ppm,no2_ppm
0,MEAS-014-000001,SENS-014,2024-01-01 00:02:10.000,2024-01-01 01:02:10.000,20.600,29.800,7.700,363.400,11.400,19.700,24.700,90.800,1.700,219.000,VALID,BGL-01_Bengkulu_AQMS.csv,NaN,NaN,NaN,NaN,NaN,NaN
1,MEAS-014-000002,SENS-014,2024-01-01 01:02:10.000,2024-01-01 02:02:10.000,20.600,32.000,7.400,335.800,11.800,19.300,24.300,89.900,2.400,183.000,VALID,BGL-01_Bengkulu_AQMS.csv,NaN,NaN,NaN,NaN,NaN,NaN
2,MEAS-014-000003,SENS-014,2024-01-01 02:02:10.000,2024-01-01 03:02:10.000,16.700,25.700,9.400,316.800,8.100,18.000,24.200,87.700,2.300,200.000,VALID,BGL-01_Bengkulu_AQMS.csv,NaN,NaN,NaN,NaN,NaN,NaN
3,MEAS-014-000004,SENS-014,2024-01-01 03:02:10.000,2024-01-01 04:02:10.000,20.000,29.500,8.300,355.100,12.400,20.600,24.200,86.500,2.700,150.000,VALID,BGL-01_Bengkulu_AQMS.csv,NaN,NaN,NaN,NaN,NaN,NaN
4,MEAS-014-000005,SENS-014,2024-01-01 04:02:10.000,2024-01-01 05:02:10.000,22.100,33.300,9.300,354.500,11.500,16.500,24.800,88.300,3.400,199.000,VALID,BGL-01_Bengkulu_AQMS.csv,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Ringkasan per file: jumlah baris dan rentang waktu
aqms_raw["_ts"] = pd.to_datetime(aqms_raw["timestamp_start"], errors="coerce")
per_file = (aqms_raw.groupby("source_file")
            .agg(baris=("measurement_id", "size"), awal=("_ts", "min"), akhir=("_ts", "max"),
                 n_sensor=("sensor_id", "nunique")))
per_file

,baris,awal,akhir,n_sensor
source_file,,,,
BGL-01_Bengkulu_AQMS.csv,24432,2024-01-01 00:02:10.000,2026-10-14 23:02:10.000,1
BGR-01_Cisarua_AQMS.csv,24432,NaT,NaT,1
BGR-02_Baranangsiang_AQMS.csv,24432,NaT,NaT,1
BJB-01_Banjarbaru_AQMS.csv,24432,2024-01-01 00:01:15.420,2026-10-14 23:01:15.420,1
BTM-01_Batam_AQMS.csv,24432,2023-12-31 23:48:15.100,2026-10-14 22:48:15.100,1
IDP-01_Indrapuri_AQMS.csv,24432,NaT,NaT,1
JKT-01_Gambir_AQMS.csv,24432,NaT,NaT,1
JKT-02_Marunda_AQMS.csv,24432,NaT,NaT,1
JKT-03_Cengkareng_AQMS.csv,24432,NaT,NaT,1


## 2. Assessing — periksa kualitas data
### 2.1 Tabel master & metadata

In [8]:
sites     = pd.read_csv(f"{DATA}/master/sites.csv")
sensors   = pd.read_csv(f"{DATA}/master/sensors.csv")
hospitals = pd.read_csv(f"{DATA}/master/hospitals.csv")
health    = pd.read_csv(f"{DATA}/health/HLT-01_health_outcomes_daily.csv")
m_dates   = pd.read_csv(f"{DATA}/metadata/metadata_dates.csv")
m_sensors = pd.read_csv(f"{DATA}/metadata/metadata_sensors.csv")
...

Ellipsis

In [9]:
for name, df in [("sites", sites), ("sensors", sensors), ("hospitals", hospitals),
                 ("m_dates", m_dates), ("m_sensors", m_sensors)]:
    audit(df, name)

===== sites | 27 baris x 27 kolom =====
Duplikat baris penuh: 0
Kolom dengan nilai kosong:


'tidak ada'

===== sensors | 27 baris x 10 kolom =====
Duplikat baris penuh: 0
Kolom dengan nilai kosong:


'tidak ada'

===== hospitals | 27 baris x 7 kolom =====
Duplikat baris penuh: 0
Kolom dengan nilai kosong:


'tidak ada'

===== m_dates | 27 baris x 6 kolom =====
Duplikat baris penuh: 0
Kolom dengan nilai kosong:


'tidak ada'

===== m_sensors | 16 baris x 6 kolom =====
Duplikat baris penuh: 0
Kolom dengan nilai kosong:


'tidak ada'

In [10]:
# Duplikat berdasarkan kunci (lebih penting daripada duplikat baris penuh)
print("sites     location_id duplikat:", sites["location_id"].duplicated().sum())
print("sensors   sensor_id duplikat  :", sensors["sensor_id"].duplicated().sum())
print("hospitals hospital_id duplikat:", hospitals["hospital_id"].duplicated().sum())

# Konsistensi penulisan teks (spasi berlebih / beda huruf besar-kecil)
for col in ["provinsi", "kota_kabupaten"]:
    print(f"\nsites.{col}: {sites[col].nunique()} nilai unik, "
          f"{sites[col].str.strip().str.lower().nunique()} setelah dinormalisasi")
print(f"hospitals.kota_kabupaten: {hospitals['kota_kabupaten'].nunique()} nilai unik, "
      f"{hospitals['kota_kabupaten'].str.strip().str.lower().nunique()} setelah dinormalisasi")

sites     location_id duplikat: 0
sensors   sensor_id duplikat  : 0
hospitals hospital_id duplikat: 0

sites.provinsi: 21 nilai unik, 21 setelah dinormalisasi

sites.kota_kabupaten: 27 nilai unik, 27 setelah dinormalisasi
hospitals.kota_kabupaten: 27 nilai unik, 27 setelah dinormalisasi


In [11]:
# Status & tanggal instalasi sensor
print(sensors["status_operasional"].value_counts(dropna=False))
sensors["tanggal_instalasi"] = pd.to_datetime(sensors["tanggal_instalasi"], errors="coerce")
print("Tanggal instalasi tidak terbaca:", sensors["tanggal_instalasi"].isna().sum())

status_operasional
Active          23
Intermittent     3
Defective        1
Name: count, dtype: int64
Tanggal instalasi tidak terbaca: 0


### 2.2 Integritas relasi antar tabel

In [12]:
print("sensor_id di AQMS tapi tidak ada di sensors :", sorted(set(aqms_raw["sensor_id"]) - set(sensors["sensor_id"]))[:20])
print("sensor di sensors tapi tidak punya data AQMS :", sorted(set(sensors["sensor_id"]) - set(aqms_raw["sensor_id"]))[:20])
print("location_id di sensors tapi tidak ada di sites:", sorted(set(sensors["location_id"]) - set(sites["location_id"]))[:20])
print("hospital_id di health tapi tidak ada di hospitals:", sorted(set(health["hospital_id"]) - set(hospitals["hospital_id"]))[:20])

norm = lambda s: s.astype(str).str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
kota_udara = set(norm(sites["kota_kabupaten"]))
kota_rs    = set(norm(hospitals["kota_kabupaten"]))
print("\nKota punya data udara tapi tidak punya RS :", sorted(kota_udara - kota_rs))
print("Kota punya RS tapi tidak punya data udara :", sorted(kota_rs - kota_udara))

sensor_id di AQMS tapi tidak ada di sensors : []
sensor di sensors tapi tidak punya data AQMS : []
location_id di sensors tapi tidak ada di sites: []
hospital_id di health tapi tidak ada di hospitals: []

Kota punya data udara tapi tidak punya RS : ['kota administrasi jakarta barat', 'kota administrasi jakarta pusat', 'kota administrasi jakarta utara']
Kota punya RS tapi tidak punya data udara : ['jakarta barat', 'jakarta pusat', 'jakarta utara']


### 2.3 AQMS 

In [13]:
aqms_raw.drop(columns="_ts", inplace=True)
audit(aqms_raw, "AQMS (mentah)")
aqms_raw.dtypes

C:\Users\Mega Agustina\AppData\Local\Temp\ipykernel_16596\2131865740.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  aqms_raw.drop(columns="_ts", inplace=True)


===== AQMS (mentah) | 667,981 baris x 22 kolom =====
Duplikat baris penuh: 0
Kolom dengan nilai kosong:


,jumlah_kosong
pm25_ug_m3,89972
pm10_ug_m3,89982
so2_ug_m3,89204
co_ug_m3,89210
o3_ug_m3,89211
no2_ug_m3,89217
temperature_c,18632
humidity_pct,18632
wind_speed_ms,18632
wind_direction_deg,14914


measurement_id            str
sensor_id                 str
timestamp_start           str
timestamp_end             str
pm25_ug_m3             object
pm10_ug_m3             object
so2_ug_m3              object
co_ug_m3               object
o3_ug_m3               object
no2_ug_m3              object
temperature_c          object
humidity_pct           object
wind_speed_ms         float64
wind_direction_deg    float64
qc_flag                   str
source_file               str
pm25_ppm              float64
pm10_ppm              float64
so2_ppm               float64
co_ppm                float64
o3_ppm                float64
no2_ppm               float64
dtype: object

In [14]:
print("Duplikat measurement_id:", aqms_raw["measurement_id"].duplicated().sum())
print("Duplikat sensor_id + timestamp_start:", aqms_raw.duplicated(["sensor_id", "timestamp_start"]).sum())
print("\nDistribusi qc_flag:")
print(aqms_raw["qc_flag"].value_counts(dropna=False))

Duplikat measurement_id: 0
Duplikat sensor_id + timestamp_start: 223

Distribusi qc_flag:
qc_flag
VALID          599185
SUSPECT         19741
MISSING         13675
CALIBRATING      8072
WARMUP           3764
PARSER_FAIL      3633
FATAL_ERR        3546
GARBAGE          3537
COMM_LOST        3433
CORRUPT          3425
SENSOR_DEAD      3389
MAINTENANCE      2581
Name: count, dtype: int64
